In [63]:
import os
import ast  # Para convertir texto en diccionarios
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.stats import friedmanchisquare, wilcoxon

import re

In [92]:
# Directorio base donde están todas las carpetas
base_dir = "/Users/neuralrehabilitationgroup/PycharmProjects/Superlets-Marina/RESULTS/PAPER/0.5S"

# --- Inicialización de variables y listas ---
match = re.match(r"(\d+(?:\.\d+)?)S", os.path.basename(base_dir))
if match:
    total_time = float(match.group(1))
    print(f"Total time encontrado: {total_time}")
else:
    print("No se encontró un número seguido de 'S' en el nombre de carpeta.")

labels = []
stft_means_f, wavelet_means_f = [], []
stft_means_t, wavelet_means_t = [], []
stft_stds_f, wavelet_stds_f = [], []
stft_stds_t, wavelet_stds_t = [], []

superlet1_means_f, superlet2_means_f, superlet3_means_f = [], [], []
superlet1_means_t, superlet2_means_t, superlet3_means_t = [], [], []
superlet1_stds_f, superlet2_stds_f, superlet3_stds_f = [], [], []
superlet1_stds_t, superlet2_stds_t, superlet3_stds_t = [], [], []

tonalidades = [0.4, 0.6, 0.8]

colores_base = {                                 
    "STFT": plt.colormaps["Blues"],              
    "Wavelet": plt.colormaps["Greens"],          
    "Superlet": plt.colormaps["Oranges"],        
}              

# --- Función auxiliar ---
def extract_frequency(folder_name):
    match = re.search(r"([\d\.]+)\s*Hz", folder_name)
    return float(match.group(1)) if match else float('inf')

# --- Filtrar y ordenar carpetas ---
sorted_folders = sorted([f for f in os.listdir(base_dir) if "Hz" in f], key=extract_frequency)

# --- Recorrer carpetas y cargar datos ---
for folder in sorted_folders:
    folder_path = os.path.join(base_dir, folder)

    if os.path.isdir(folder_path):
        filename_1 = os.path.join(folder_path, "maes_f.txt")
        filename_2 = os.path.join(folder_path, "maes_t.txt")

        if os.path.exists(filename_1) and os.path.exists(filename_2):
            with open(filename_1, "r", encoding="utf-8") as file:
                maes_f = ast.literal_eval(file.read())

            with open(filename_2, "r", encoding="utf-8") as file:
                maes_t = ast.literal_eval(file.read())

            labels.append(folder)
            
            stft_means_f.append(maes_f["stft"][1][1])
            wavelet_means_f.append(maes_f["wavelet"][1][1])
            stft_stds_f.append(maes_f["std_stft"][1][1])
            wavelet_stds_f.append(maes_f["std_wavelet"][1][1])
            
            stft_means_t.append(maes_t["stft"][1][1])
            wavelet_means_t.append(maes_t["wavelet"][1][1])
            stft_stds_t.append(maes_t["std_stft"][1][1])
            wavelet_stds_t.append(maes_t["std_wavelet"][1][1])

            # Superlet (3 parámetros)
            superlet1_means_f.append(maes_f["superlet"][1][0])
            superlet2_means_f.append(maes_f["superlet"][1][1])
            superlet3_means_f.append(maes_f["superlet"][1][2])
            superlet1_stds_f.append(maes_f["std_superlet"][1][0])
            superlet2_stds_f.append(maes_f["std_superlet"][1][1])
            superlet3_stds_f.append(maes_f["std_superlet"][1][2])

            superlet1_means_t.append(maes_t["superlet"][1][0])
            superlet2_means_t.append(maes_t["superlet"][1][1])
            superlet3_means_t.append(maes_t["superlet"][1][2])
            superlet1_stds_t.append(maes_t["std_superlet"][1][0])
            superlet2_stds_t.append(maes_t["std_superlet"][1][1])
            superlet3_stds_t.append(maes_t["std_superlet"][1][2])

Total time encontrado: 0.5


In [93]:
stft_f = np.array(stft_means_f)
cwt_f = np.array(wavelet_means_f)

slt1_f = np.array(superlet1_means_f)
slt2_f = np.array(superlet2_means_f)
slt3_f = np.array(superlet3_means_f)

stft_t = np.array(stft_means_t)
cwt_t = np.array(wavelet_means_t)

slt1_t = np.array(superlet1_means_t)
slt2_t = np.array(superlet2_means_t)
slt3_t = np.array(superlet3_means_t)


In [94]:
def wilcoxon_effect(x, y):
    stat, p = wilcoxon(x, y)
    n = len(x)
    mean_w = n*(n+1)/4
    std_w = np.sqrt(n*(n+1)*(2*n+1)/24)
    z = (stat - mean_w) / std_w
    r = abs(z) / np.sqrt(n)
    return p, r

def reduction(baseline, method):
    return np.mean((baseline - method) / baseline * 100)

def create_table(baseline_t, baseline_f, slt_t_list, slt_f_list, slt_labels, baseline):
    """Crea dos DataFrames: MAE_t y MAE_f"""
    # Temporal error
    rows_t = []
    mean_baseline_t = np.mean(baseline_t)
    std_baseline_t = np.std(baseline_t)
    
    for name, data_t in zip(slt_labels, slt_t_list):
        row = {}
        row["Method"] = name
        row[f"Mean ± STD ({baseline})"] = f"{mean_baseline_t:.3f} ± {std_baseline_t:.3f}"
        row["Mean ± STD (SLT)"] = f"{np.mean(data_t):.3f} ± {np.std(data_t):.3f}"
        row[f"p vs {baseline}"], row[f"r vs {baseline}"] = wilcoxon_effect(data_t, baseline_t)
        row[f"Reduction vs {baseline} (%)"] = reduction(baseline_t, data_t)
        rows_t.append(row)

    df_t = pd.DataFrame(rows_t)

    # Frequency error
    rows_f = []
    mean_baseline_f = np.mean(baseline_f)
    std_baseline_f = np.std(baseline_f)
    
    for name, data_f in zip(slt_labels, slt_f_list):
        row = {}
        row["Method"] = name
        row[f"Mean ± STD ({baseline})"] = f"{mean_baseline_f:.3f} ± {std_baseline_f:.3f}"
        row["Mean ± STD (SLT)"] = f"{np.mean(data_f):.3f} ± {np.std(data_f):.3f}"
        row[f"p vs {baseline}"], row[f"r vs {baseline}"] = wilcoxon_effect(data_f, baseline_f)
        row[f"Reduction vs {baseline} (%)"] = reduction(baseline_f, data_f)
        rows_f.append(row)

    df_f = pd.DataFrame(rows_f)
    return df_t, df_f

# --- listas de SLT ---
slt_labels = ["SLT1", "SLT2", "SLT3"]
slt_t_list = [slt1_t, slt2_t, slt3_t]
slt_f_list = [slt1_f, slt2_f, slt3_f]

df_t_stft, df_f_stft = create_table(stft_t, stft_f, slt_t_list, slt_f_list, slt_labels, baseline='STFT')
df_t_cwt, df_f_cwt = create_table(cwt_t, cwt_f, slt_t_list, slt_f_list, slt_labels, baseline = 'CWT')


df_t_stft.to_csv(os.path.join(base_dir,"MAE_t_results_STFT.csv"), index=False)
df_f_stft.to_csv(os.path.join(base_dir,"MAE_f_results_STFT.csv"), index=False)
df_t_cwt.to_csv(os.path.join(base_dir,"MAE_t_results_CWT.csv"), index=False)
df_f_cwt.to_csv(os.path.join(base_dir,"MAE_f_results_CWT.csv"), index=False)

In [95]:
df_t_stft.head()

,Method,Mean ± STD (STFT),Mean ± STD (SLT),p vs STFT,r vs STFT,Reduction vs STFT (%)
0,SLT1,0.250 ± 0.102,0.221 ± 0.117,0.015625,0.894427,15.939167
1,SLT2,0.250 ± 0.102,0.237 ± 0.110,0.031250,0.830540,7.452719
2,SLT3,0.250 ± 0.102,0.331 ± 0.089,0.015625,0.894427,-38.639856


In [96]:
df_f_stft.head()

,Method,Mean ± STD (STFT),Mean ± STD (SLT),p vs STFT,r vs STFT,Reduction vs STFT (%)
0,SLT1,3.594 ± 0.433,5.494 ± 0.357,0.015625,0.894427,-54.384034
1,SLT2,3.594 ± 0.433,4.011 ± 0.489,0.015625,0.894427,-12.049577
2,SLT3,3.594 ± 0.433,4.627 ± 2.531,0.375000,0.383326,-28.578024


In [97]:
df_t_cwt.head()

,Method,Mean ± STD (CWT),Mean ± STD (SLT),p vs CWT,r vs CWT,Reduction vs CWT (%)
0,SLT1,0.260 ± 0.148,0.221 ± 0.117,0.031250,0.830540,11.851525
1,SLT2,0.260 ± 0.148,0.237 ± 0.110,0.296875,0.447214,1.781503
2,SLT3,0.260 ± 0.148,0.331 ± 0.089,0.078125,0.702764,-52.132016


In [98]:
df_f_cwt

,Method,Mean ± STD (CWT),Mean ± STD (SLT),p vs CWT,r vs CWT,Reduction vs CWT (%)
0,SLT1,7.044 ± 0.734,5.494 ± 0.357,0.015625,0.894427,21.089623
1,SLT2,7.044 ± 0.734,4.011 ± 0.489,0.015625,0.894427,42.559337
2,SLT3,7.044 ± 0.734,4.627 ± 2.531,0.046875,0.766652,34.534322
